## Script to parallelize the forward pass across the population (new version because some functions were deprecated in recent pytorch ...)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
sys.path.append('C:/Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils')

from NN_utils import *
import torch
import torch.nn as nn
from torchvision import datasets, transforms
#from torchsummary import summary
import time
from types import SimpleNamespace
import pickle
import gc
from optimization_algorithms import *

from torch.func import functional_call
from torch import vmap
from collections import OrderedDict


In [2]:
transform_data = transforms.Compose([
    transforms.ToTensor()
    #transforms.Normalize((0.2868,), (0.3524,))
])

MNIST_train = datasets.MNIST(root='./data', train=True, transform=transform_data, download=True)
MNIST_test = datasets.MNIST(root='./data', train=False, transform=transform_data, download=True)

train_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_train, batch_size=1000, shuffle=True)
test_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_test, batch_size=10000, shuffle=False)

X_train_MNIST, Y_train_MNIST = next(iter(train_loader_MNIST))
X_test_MNIST, Y_test_MNIST = next(iter(test_loader_MNIST))

In [3]:
n_neurons = 100

RNN_params = {
        "N_in": 784,               # e.g., flattened 28x28 FashionMNIST image
        "N_out": 10,               # number of classes in FashionMNIST
        "N_neurons": n_neurons,          # number of hidden units per RNN layer
        "N_layers": 3,             # depth of the RNN
        "time_steady_state": 200    # number of repeated timesteps to reach steady state
    }
    
model = Oscillator_RNN_dyn(params=RNN_params).float()
    
model.init_esn_weights(reservoir = True)
model.dt = 0.1
model.eps_int = 1e-4
model.alpha=3
model.max_steps=40
model.save_activations = False

N_dim = model.count_parameters()

loss = nn.CrossEntropyLoss()
# learning parameters

init_pos = model.get_params()

if init_pos.requires_grad:
    # Detach the tensor from the computation graph
    init_pos = init_pos.detach()
if init_pos.is_cuda:
    # Move the tensor to the CPU
    init_pos = init_pos.cpu()
init_pos = init_pos.numpy()

pop_size = int(0.01*N_dim)
PEPG_optimizer = PEPG_opt(N_dim, pop_size = pop_size, learning_rate=0.01, starting_mu=init_pos ,starting_sigma=1e-1)

PEPG_optimizer.sigma_decay = 0.9999
PEPG_optimizer.sigma_alpha=0.2
PEPG_optimizer.sigma_limit=0.01
PEPG_optimizer.elite_ratio=0.1
PEPG_optimizer.weight_decay=0.005
        
#print(f'Using {n_neurons} per layer, run {s+1}, number of parameters {model.count_parameters()}')
#D = train_online_pop_NN(model, n_epochs, train_loader_MNIST, test_loader_MNIST, loss, PEPG_optimizer)
#results.append(D)

In [4]:
def build_batched_state_dict(coord, model, dtype=torch.float32, device=None):
    """
    Convert a population of flattened trainable parameters into one batched state_dict
    suitable for torch.func.functional_call + torch.vmap.

    Parameters:
    - coord: np.ndarray of shape (pop_size, n_trainable_params)
    - model: your nn.Module (already moved to device & in correct dtype)
    - dtype: desired torch.dtype for trainable params (default torch.float32)
    - device: torch.device to place all tensors on (e.g. 'cuda')

    Returns:
    - batched_state: OrderedDict where each key maps to a tensor of shape
      (pop_size, *original_shape)
    """

    # 1) Reference state_dict (includes both params & buffers)
    ref_state = model.state_dict()  # OrderedDict of name -> Tensor

    # 2) Extract which parameters are trainable
    named_params = dict(model.named_parameters())
    trainable_keys = [k for k, p in named_params.items() if p.requires_grad]

    # 3) Build shapes & sizes for trainable slices, in the order of trainable_keys
    shapes = [tuple(ref_state[k].shape) for k in trainable_keys]
    sizes  = [int(np.prod(s)) for s in shapes]
    cum    = np.cumsum([0] + sizes)

    # Sanity check: coord width must match total trainable size
    total_trainable = cum[-1]
    assert coord.shape[1] == total_trainable, \
        f"coord has {coord.shape[1]} values but expected {total_trainable}"

    pop_size = coord.shape[0]

    # 4) For each member of the population, build a full state_dict
    all_states = []
    for i in range(pop_size):
        flat = coord[i]
        state = OrderedDict()
        # track index into flat vector
        idx = 0

        for key, tensor in ref_state.items():
            if key in trainable_keys:
                # unflatten the next slice
                start, end = cum[idx], cum[idx+1]
                chunk = torch.tensor(
                    flat[start:end], dtype=dtype, device=device
                ).view(shapes[idx])
                state[key] = chunk
                idx += 1
            else:
                # fixed param or buffer: copy original (already on device if you moved model)
                state[key] = tensor.to(device)
        all_states.append(state)

    # 5) Stack across the population to get batched_state
    batched_state = OrderedDict()
    for key in ref_state.keys():
        # collect [pop_size] tensors and stack
        stacked = torch.stack([st[key] for st in all_states], dim=0)
        batched_state[key] = stacked

    return batched_state

In [6]:
def build_batched_state_dict_fast(coord, model, dtype=torch.float32, device=None):
    """
    Vectorized version: builds one batched state_dict (key→Tensor of shape [pop_size, …])
    without Python loops over the population dimension.

    Parameters:
    - coord: np.ndarray, shape (pop_size, n_trainable_params)
    - model: nn.Module (already moved to device & dtype)
    - dtype: torch.dtype (default torch.float32)
    - device: torch.device (e.g. 'cuda')

    Returns:
    - batched_state: OrderedDict[key → Tensor of shape (pop_size, *orig_shape)]
    """
    pop_size, n_train = coord.shape

    # 1) Grab the reference state (params + buffers)
    ref_state = model.state_dict()  # OrderedDict

    # 2) Identify trainable parameter names & compute their flat offsets
    named_params = dict(model.named_parameters())
    trainable_keys = [k for k,p in named_params.items() if p.requires_grad]

    # For each trainable key, record shape and size
    shapes = [tuple(ref_state[k].shape) for k in trainable_keys]
    sizes  = [int(np.prod(s)) for s in shapes]
    cum    = np.cumsum([0] + sizes)  # offsets into flat vector

    # 3) Sanity check
    assert cum[-1] == n_train, f"coord has {n_train} but need {cum[-1]}"

    # 4) Build the new batched state_dict
    batched = OrderedDict()

    # (a) First fill in trainable params by slicing coord in one go
    for idx, key in enumerate(trainable_keys):
        start, end = cum[idx], cum[idx+1]
        # slice out (pop_size, size)
        slice_np = coord[:, start:end]  # numpy
        # convert & reshape in one shot: (pop_size, *orig_shape)
        batched[key] = (
            torch
            .from_numpy(slice_np.astype(np.float32))
            .to(device)
            .view(pop_size, *shapes[idx])
        )

    # (b) Then fill in frozen params & buffers by expanding the single tensor
    for key, tensor in ref_state.items():
        if key in trainable_keys:
            continue
        # ensure it’s on correct device & dtype
        t = tensor.to(device=device, dtype=dtype)
        # expand to (pop_size, *orig_shape)
        batched[key] = t.unsqueeze(0).expand(pop_size, *t.shape)

    return batched

In [7]:
# Get candidate parameter matrix (float32)
coordinates = PEPG_optimizer.ask()

In [8]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)

# Make functional and move buffers/params to device


# Vectorized model
batched_forward = vmap(
    lambda state, x: functional_call(model, state, (x,)),
    in_dims=(0, None),
)

# Move input to device
X_train_MNIST = X_train_MNIST.float().to(device)

# Build batched parameters on GPU
start_time = time.time()
# Forward pass on GPU
with torch.no_grad():
    
    batched_params = build_batched_state_dict_fast(coordinates, model, dtype=torch.float32, device=device)

    Y_pred_parallel = batched_forward(batched_params, X_train_MNIST)

end_time = time.time()
print(f'Parallel forward pass time: {end_time - start_time:.4f} seconds')

Using device: cuda
Parallel forward pass time: 0.5879 seconds


In [9]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)


# Move input to device
X_train_MNIST = X_train_MNIST.float().to(device)

# Build batched parameters on GPU
Y_pred = torch.zeros([coordinates.shape[0],1000,10],device=device)
start_time = time.time()

# Forward pass on GPU
with torch.no_grad():
    for k in range(coordinates.shape[0]):
                    
        Y_pred[k,:,:] = model.forward_pass_params(coordinates[k,:],X_train_MNIST)

end_time = time.time()
print(f'Regular forward pass time: {end_time - start_time:.4f} seconds')

Using device: cuda
Regular forward pass time: 20.3368 seconds


In [12]:
4.7/0.2

23.5

In [10]:
diff_outputs = 100*torch.abs(Y_pred_parallel-Y_pred)/Y_pred

In [11]:
torch.mean(diff_outputs)

tensor(-5.4743e-06, device='cuda:0')

In [13]:
def population_cross_entropy_loss(population_output, labels):
    """
    Compute average cross-entropy loss per population member.

    Parameters:
    - population_output: Tensor of shape (pop_size, batch_size, n_classes)
    - labels: Tensor of shape (batch_size,) with target class indices

    Returns:
    - population_losses: Tensor of shape (pop_size,) with mean loss per individual
    """

    pop_size, batch_size, n_classes = population_output.shape

    # Repeat labels across population
    labels_expanded = labels.unsqueeze(0).expand(pop_size, -1)  # (pop_size, batch_size)

    # Flatten both for loss computation
    outputs_flat = population_output.reshape(-1, n_classes)     # (pop_size * batch_size, n_classes)
    labels_flat = labels_expanded.reshape(-1)                   # (pop_size * batch_size,)

    # Compute per-sample loss without reduction
    losses_flat = F.cross_entropy(outputs_flat, labels_flat, reduction='none')  # (pop_size * batch_size,)

    # Reshape back to (pop_size, batch_size)
    losses = losses_flat.view(pop_size, batch_size)

    # Mean loss per individual
    population_losses = losses.mean(dim=1)  # (pop_size,)

    return population_losses

In [ ]:
Y_train_MNIST = Y_train_MNIST.to(device)
Loss_pop = population_cross_entropy_loss(Y_pred_parallel,Y_train_MNIST)